In [6]:
import datasets
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import numpy as np
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import nltk
from sentence_transformers import SentenceTransformer, util
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments
from torch.utils.data import Subset
from tqdm import tqdm
import os
import random

2026-02-14 17:47:20.178365: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771091240.200710     212 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771091240.207557     212 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771091240.224720     212 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771091240.224743     212 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771091240.224745     212 computation_placer.cc:177] computation placer alr

In [8]:
dataset = load_dataset("IlyaGusev/gazeta")
model_name = "google/mt5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
bert_model = SentenceTransformer("ai-forever/sbert_large_mt_nlu_ru").to(device)
class DataPreparation():
    def __init__(self, sentences_distribution, max_input_len, device):
        """
        Args:
            sentences_distribution (dict): (keys: small, med, big). Dictionary specifying how many sentences to select for each text length category.
            max_input_len (int): Maximum input sequence length for the tokenizer.
            device (str): Device to run embeddings and tensors on ('cuda' or 'cpu').
        """
        super().__init__()
        
        self.device = device
        self.tokenizer = tokenizer
        self.tokenizer.truncation_side = 'right'
        
        self.sentences_distribution = sentences_distribution
        self.max_input_len = max_input_len
        
    
    def _sentence_emb(self, sentence):
        """
        Compute sentence embeddings using the pre-loaded BERT model.
        Args:
            sentence (str or List[str]): Sentence or list of sentences to embed.
        Returns:
            torch.Tensor: Embedding vector(s) for the input sentence(s)(or text) on the device.
        """
        with torch.no_grad():
            return bert_model.encode(sentence, convert_to_tensor=True)
    
    def _devide_text_on_chunks(self, text, sent_in_chunk):
        """
        Split text into chunks of sentences and score each sentence by similarity to the chunk-level summary embedding.
        Args:
            text (str): The full text to split and score.
            sent_in_chunk (int): Number of sentences per chunk.
        Returns:
            List[List[Tuple[int, str, float]]]: A list of chunks, where each chunk is a list of tuples:
                (original_sentence_index, sentence_text, similarity_score)
        """
        sentences = nltk.sent_tokenize(text)
        chunks = [[]]

        for i, s in enumerate(sentences):
            if len(chunks[-1]) < sent_in_chunk:
                chunks[-1].append((i, s))
            else:
                chunks.append([(i, s)])

        summary_embedings = []
        for cur_chunk in chunks:
            cur_text = [" ".join([cur[1] for cur in cur_chunk])]
            summary_embedings.append(self._sentence_emb(cur_text).reshape(-1))
        
        scored_chunks = []
        for cur_summary_emb, chunk in zip(summary_embedings, chunks):
            s_texts = [c[1] for c in chunk]
            emb_sents = self._sentence_emb(s_texts)  # (m, d)
            
            sims = util.cos_sim(emb_sents, cur_summary_emb.to(self.device).unsqueeze(0)).squeeze(1)  # (m,)
            
            scored = []
            for (orig_idx, sent), score in zip(chunk, sims):
                scored.append((orig_idx, sent, float(score.item())))
            scored_chunks.append(scored)
        return scored_chunks

    def _get_need_sentences(self, text, sentences_distribution):
        """
        Select important sentences from the text based on chunk similarity scores
        and the sentences_distribution configuration.
        Args:
            text (str): The full input text.
            sentences_distribution (dict): Dictionary with keys:
                'best_sbert' - top sentences to select by similarity
                'worst_sbert' - least similar sentences to include
                'random' - additional random sentences from remaining ones
        Returns:
            List[str]: List of selected sentences, sorted by their original order in text.
        """
        raw_chunks = self._devide_text_on_chunks(text, sentences_distribution['sent_in_chunk'])
        
        all_need_sentences = []
        for chunk in raw_chunks:
            chunk_sorted = sorted(chunk, key=lambda cur: cur[2], reverse = True)
        
            need_sentences = []
            best = sentences_distribution['best_sbert']
            worst = sentences_distribution['worst_sbert']
            rand_n = sentences_distribution['random']

            
            for i in range(min(len(chunk_sorted), best)):
                need_sentences.append(chunk_sorted[i])
            for i in range(max(0, min(len(chunk_sorted) - best, worst))):
                need_sentences.append(chunk_sorted[- i - 1])
                
            for _ in range(max(0, min(len(chunk_sorted) - best - worst, rand_n))):
                cur_idx = torch.randint(
                    low = best, 
                    high = len(chunk_sorted) - worst,
                    size=(1,)
                )[0].item()
                need_sentences.append(chunk_sorted[cur_idx])
            need_sentences = sorted(need_sentences, key = lambda cur: cur[0])
            need_sentences = [cur[1] for cur in need_sentences]
            
            all_need_sentences += need_sentences
            
        return all_need_sentences

    def _tokenize_text(self, text, add_special_tokens: bool):
        """
        Tokenize a text string using the tokenizer with optional special tokens.
        Args:
            text (str): The input text to tokenize.
            add_special_tokens (bool): Whether to add special tokens
        Returns:
            Tuple[torch.Tensor, torch.Tensor]: input_ids and attention_mask tensors.
        """
        out_token = self.tokenizer(
            text,
            max_length=self.max_input_len,
            truncation=True,   
            padding=False,
            return_tensors='pt',
            add_special_tokens = add_special_tokens,
        )
        return out_token['input_ids'], out_token['attention_mask']
    
    def prepare(self, text):
        """
        Prepare the input text for the seq2seq summarization model.

        Steps:
        1. Determine text length category (small/medium/big) and select sentences.
        2. Build the prompt by concatenating selected sentences with a summary instruction.
        3. Tokenize the prompt and ending text separately.
        4. Concatenate input_ids and create attention mask.

        Args:
            text (str): Original text to summarize.
        Returns:
            dict: {
                    'text': original text,
                'input_ids': tensor of token IDs (concatenated prompt + ending),
                'attention_mask': tensor of attention mask,
                'need_text': concatenated prompt text used as model input
            }
        """
        cur_len = len(self.tokenizer(text)['input_ids'])
        if (cur_len <= 1100):
            sentences_distribution = self.sentences_distribution['small']
        elif (cur_len <= 1600):
            sentences_distribution = self.sentences_distribution['med']
        else:
            sentences_distribution = self.sentences_distribution['big']
        
        need_sentences = self._get_need_sentences(text, sentences_distribution)
        need_sentences = " ".join(need_sentences)
        
        in_text = "Сформулируй краткое содержание:\n\n" + need_sentences        
        ending_text = "...\n\nКраткое содержание:" + self.tokenizer.eos_token
        
        in_text_ids, _ = self._tokenize_text(in_text, add_special_tokens = False)
        ending_ids, _ = self._tokenize_text(ending_text, add_special_tokens = False)
        in_text_ids = in_text_ids.squeeze(0)
        ending_ids = ending_ids.squeeze(0)
    
        input_ids = torch.cat((in_text_ids, ending_ids), dim=0)
        attn_mask = torch.ones_like(input_ids)
        
        ans = {
            'text': text,
            'input_ids': input_ids,
            'attention_mask': attn_mask,
            'need_text': in_text + ending_text
        }
        return ans

In [10]:

sentences_distribution = {
    'small': { # <= 1100 tokens
        'sent_in_chunk': 11,
        'best_sbert': 2,
        'worst_sbert': 1,
        'random': 2,
    }, 
    'med': { # <= 1600 tokens
        'sent_in_chunk': 15,
        'best_sbert': 2,
        'worst_sbert': 1,
        'random': 2,
    },
    'big': { # > 1600 tokens
        'sent_in_chunk': 17,
        'best_sbert': 2,
        'worst_sbert': 1,
        'random': 2,
    }
}
max_input_len = 504
device = 'cuda'
data_prep = DataPreparation(sentences_distribution, max_input_len, device)

In [13]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

base_model_name = "google/mt5-large"
checkpoint_path = "/kaggle/input/models/mcaramba563/4-models/pytorch/default/1/results_model/checkpoint-2000"

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    base_model_name,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, checkpoint_path)
model.eval()
print()

model.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'peft_version'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


In [ ]:
def generate_answer(text):
    """
    Generate a summary for the given input text
    Args:
        text (str): The original text to summarize.
    Returns:
        str: The generated summary text.
    """
    sample = data_prep.prepare(text)

    inputs = {
        'input_ids': sample['input_ids'].unsqueeze(0).to(model.device),
        'attention_mask': sample['attention_mask'].unsqueeze(0).to(model.device)
    }
    
    with torch.no_grad():
        
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            early_stopping=True
        )
    out_text = tokenizer.decode(out[0], skip_special_tokens=True)
    return out_text

## Fine-Tuned Model

In [20]:
print(generate_answer(dataset['test'][1]['text']))

В ОАЭ высокопоставленная американская и израильская делегация находятся в ОАЭ с двухдневным визитом, за время которого стороны заключили историческое соглашение о нормализации отношений между США, Израилем и ОАЭ.


In [16]:
print(generate_answer(dataset['test'][100]['text']))

Вице-премьер и экс-посол Украины в Белоруссии Роман Бессмертный предсказал новый «майдан» и потерю власти действующему президенту Украины Владимиру Зеленскому. Он заявил, что Украина близится к тому, чтобы стать парламентской республикой, а Зеленский может оказаться последним президентом страны.


In [17]:
print(generate_answer(dataset['test'][4]['text']))

В России вступают в силу поправки в закон «О банкротстве» — теперь должники смогут освобождаться от непосильных обязательств во внесудебном порядке, если сумма задолженности составляет не менее 50 тыс. рублей.


In [18]:
print(generate_answer(dataset['test'][432]['text']))

В Выборге задержан председатель районного комитета финансов Александр Болучевский — его подозревают в хищении 680 млн рублей. По словам главы администрации Выборгского района Геннадия Орлова, его подозревают в хищении 680 млн рублей.


## Default Model

In [21]:
model_name = "google/mt5-large"
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    device_map="auto",
)

In [22]:
print(generate_answer(dataset['test'][1]['text']))

<extra_id_0> и Израилем. Краткое содержание: <extra_id_1> и Израиля.  <extra_id_2> и Израиля.  <extra_id_3> и Израиля.  <extra_id_4> и Израиля.  <extra_id_5> и Израиля.  <extra_id_6> и Израиля.  <extra_id_7> и Израиля.  <extra_id_8>.  <extra_id_9>.  <extra_id_10>.  <extra_id_11>.  <extra_id_12>.  <extra_id_13>.  <extra_id_14>.  <extra_id_15>.  <extra_id_16>.  <extra_id_17>.  <extra_id_18>.  <extra_id_19>.  <extra_id_20>.  <extra_id_21>.  <extra_id_22>.  <extra_id_23>.  <extra_id_24>.  <extra_id_25>.  <extra_id_26>.  <extra_id_27>.


In [23]:
print(generate_answer(dataset['test'][100]['text']))

<extra_id_0> президента Украины Владимира Зеленского.  <extra_id_1> президента Украины Владимира Зеленского.  <extra_id_2> президента Украины.  <extra_id_3> президента Украины.  <extra_id_4> президента Украины.  <extra_id_5> президента Украины.  <extra_id_6> президента Украины.  <extra_id_7> президента Украины.  <extra_id_8> президента Украины.  <extra_id_9> президента Украины.  <extra_id_10> президента Украины.  <extra_id_11> президента.  <extra_id_55> президента. .  <extra_id_56> президент


In [24]:
print(generate_answer(dataset['test'][4]['text']))

<extra_id_0> краткое содержание: ... <extra_id_1> краткое содержание: ... Краткое содержание: ... Краткое содержание: ... Краткое содержание: ... <extra_id_2> краткое содержание: ... <extra_id_3> краткое содержание: ... <extra_id_4> краткое содержание: ... <extra_id_5> краткое содержание: ... <extra_id_6>: ... <extra_id_7>: ... <extra_id_8>: ... <extra_id_21>: ... <extra_id_22>: ... <extra_id_23>: ... <extra_id_24>: ... <extra_id_25>. <extra_id_26>.  <extra_id_27>.  <extra_id_28>.  <extra_id_29>.  <extra_id_30>.  <extra_id_31>.  <extra_id_32>.  <extra_id_33>.  <extra_id_34>.  <extra_id_35>.  <extra_id_36>.  <extra_id_37>. 


In [27]:
print(generate_answer(dataset['test'][432]['text']))

<extra_id_0> и подрядчик. Краткое содержание: В Выборге задержан председатель районного комитета финансов Александр Болучевский. <extra_id_1>.  <extra_id_2>.  <extra_id_3>.  <extra_id_4>.  <extra_id_5>.  <extra_id_6>.  <extra_id_7>.  <extra_id_8>.  <extra_id_9>.  <extra_id_10>.  <extra_id_11>.  <extra_id_12>.  <extra_id_13>.  <extra_id_14>.  <extra_id_15>.  <extra_id_16>.  <extra_id_17>.  <extra_id_18>.  <extra_id_19>.  <extra_id_20>.  <extra_id_21>.  <extra_id_22>.  <extra_id_23>.  <extra_id_24>.  <extra_id_25>.  <extra_id_26>.  <extra_id_27>.  <extra_id_28>.  <extra_id_29>.  <extra_id_30>.  <extra_id_31>.  <extra_id_32>.
